# garak against a small model you run yourself

Runs in Google Colab or a GitHub Codespace, in your browser. Nothing is installed on your computer.

> **Scope.** Run this only against models and apps you own or the toy targets defined here. No third-party systems, no real people's data. Everything in this repository is synthetic.

garak is NVIDIA's LLM vulnerability scanner (https://github.com/NVIDIA/garak). This notebook pins garak 0.17.1 and scans `gpt2`, a small open model that runs on the free CPU runtime. It is a weak model, so expect many hits: the point is to learn the workflow, read the report and compute attack success rates.

In [ ]:
%pip install -q garak==0.17.1

List the probes this version ships, then run one probe family with a few generations per prompt.

In [ ]:
!python -m garak --list_probes | head -40

In [ ]:
!python -m garak --target_type huggingface --target_name gpt2 --probes encoding --generations 2

garak prints where it wrote its JSONL report. Load it and compute the attack success rate per probe with a Wilson interval (Module 16).

In [ ]:
import glob, json, math, os, collections

def wilson(x, n, z=1.96):
    if n == 0:
        return (0.0, 1.0)
    p = x / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h

reports = sorted(glob.glob(os.path.expanduser("~/.local/share/garak/garak_runs/*.report.jsonl")), key=os.path.getmtime)
print(reports[-1] if reports else "No report found: check the path garak printed above.")
counts = collections.defaultdict(lambda: [0, 0])
if reports:
    for line in open(reports[-1]):
        r = json.loads(line)
        if r.get("entry_type") == "eval":
            counts[(r["probe"], r["detector"])][0] += r["fails"]
            counts[(r["probe"], r["detector"])][1] += r["total_evaluated"]
for (probe, det), (fails, total) in sorted(counts.items()):
    lo, hi = wilson(fails, total)
    print(f"{probe:45s} {det:35s} {fails:4d}/{total:<4d} ASR={fails/total:.2f} 95% CI {lo:.2f}-{hi:.2f}")